# Example 01 — Iris Classification

This notebook demonstrates the basic workflow for training and evaluating
a `FuzzyARTMAP` classifier on the classic Iris dataset.

**Key concepts covered:**
- Normalisation to `[0, 1]`
- Training with `fit()`
- Inspecting committed nodes
- Evaluating with sklearn metrics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

from fuzzyart import FuzzyARTMAP
from fuzzyart.preprocessing import normalize

## 1. Load and preprocess data

In [ ]:
X, y = load_iris(return_X_y=True)

# FuzzyARTMAP requires all inputs in [0, 1]
X = normalize(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

## 2. Train the classifier

In [ ]:
clf = FuzzyARTMAP(
    alpha=0.01,        # signal rule parameter
    beta=0.5,          # learning rate
    rho_baseline=0.0,  # baseline vigilance
    epochs=10,
    verbose=True,
)
clf.fit(X_train, y_train)
print(f'\nCommitted nodes: {clf.n_committed_}')

## 3. Evaluate

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=load_iris().target_names))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=load_iris().target_names,
    ax=ax, colorbar=False
)
ax.set_title('FuzzyARTMAP — Iris')
plt.tight_layout()
plt.show()

## 4. Inspect the model

In [ ]:
import pandas as pd

summary = clf.summary()
print(pd.Series({
    'Committed nodes': summary['n_committed'],
    'Input features': summary['n_features'],
    'Classes': summary['n_classes'],
}))

# Nodes per class
labels = clf.get_node_labels()
for c in sorted(set(labels)):
    print(f'  Class {c}: {labels.count(c)} nodes')

## 5. Effect of vigilance on compression

Higher `rho_baseline` → stricter matching → more committed nodes.

In [ ]:
rhos = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9]
node_counts = []
accuracies = []

for rho in rhos:
    c = FuzzyARTMAP(rho_baseline=rho, epochs=10).fit(X_train, y_train)
    node_counts.append(c.n_committed_)
    accuracies.append(np.mean(c.predict(X_test) == y_test))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(rhos, node_counts, 'o-')
ax1.set_xlabel('rho_baseline'); ax1.set_ylabel('Committed nodes'); ax1.set_title('Nodes vs Vigilance')
ax2.plot(rhos, accuracies, 's-', color='green')
ax2.set_xlabel('rho_baseline'); ax2.set_ylabel('Test accuracy'); ax2.set_title('Accuracy vs Vigilance')
plt.tight_layout(); plt.show()